![](https://raw.githubusercontent.com/Carl-McBride-Ellis/images_for_kaggle/main/H2O_ai_logo.png)
# H2O.ai Gradient boosting classifier
In this notebook we shall be using the Gradient boosting classifier ([`H2OGradientBoostingEstimator`](https://docs.h2o.ai/h2o/latest-stable/h2o-docs/data-science/gbm.html)) from [H2O.ai](https://www.h2o.ai/)

To learn more about H2O.ai see:
* [H2O.ai Overview](https://docs.h2o.ai/h2o/latest-stable/h2o-docs/index.html)
* [H2O.ai Tutorials](https://docs.h2o.ai/h2o-tutorials/latest-stable/index.html)

Firstly, import `h2o` and start a local H2O server

In [ ]:
import h2o
h2o.init()

Read in the data as a [H2OFrame](https://docs.h2o.ai/h2o/latest-stable/h2o-py/docs/frame.html), the primary data store for H2O. For examples of munging with H2O see the [Data Manipulation](https://docs.h2o.ai/h2o/latest-stable/h2o-docs/data-munging.html) page.

In [ ]:
train_data = h2o.import_file('../input/titanic/train.csv')
test_data  = h2o.import_file('../input/titanic/test.csv')

Convert the `Survived` column in the `train_data` frame to be categorical, indicating to the estimator that this is a classification problem. As for the other categorical features, H2O automatically takes care of them.

In [ ]:
train_data["Survived"] = train_data["Survived"].asfactor()

Let us take a look at the `train_data`

In [ ]:
train_data

create a list of the features we are interested in

In [ ]:
predictors = ["Pclass", "Sex", "SibSp", "Parch", "Embarked", "Age", "Fare"]
target     = "Survived"

We shall be using the [H2O Gradient Boosting Machine](https://docs.h2o.ai/h2o/latest-stable/h2o-docs/data-science/gbm.html). Note that it is [extremely easy to overfit the Titanic dataset](https://www.kaggle.com/carlmcbrideellis/overfitting-and-underfitting-the-titanic), so we shall only use 15 trees, having a maximum depth of 4

In [ ]:
from h2o.estimators.gbm import H2OGradientBoostingEstimator

classifier  =  H2OGradientBoostingEstimator(nfolds =    5,
                                            ntrees =   15,
                                            seed   =    1,
                                            max_depth = 4)

classifier.train(predictors, target, training_frame = train_data)

Let us take a look at the predictions

In [ ]:
predictions = classifier.predict(test_data)
predictions

we are interested in the `p1` class, which we shall convert from a probability to a binary classification, here setting the [discrimination threshold](https://www.kaggle.com/carlmcbrideellis/discrimination-threshold-false-positive-negative) to be 0.5

In [ ]:
discrimination_threshold = 0.5
Survived = ((predictions["p1"] > discrimination_threshold )*1).set_names(['Survived'])

we shall now [combine](https://docs.h2o.ai/h2o/latest-stable/h2o-docs/data-munging/combining-columns.html) our class predictions with the `test_data`

In [ ]:
test_with_predictions = test_data.cbind(Survived)
test_with_predictions

# Create a `submission.csv` for scoring by kaggle

In [ ]:
submission = test_with_predictions[:,["PassengerId","Survived"]]
h2o.export_file(submission, path = "submission.csv", force = True)

We have finished, and shall now shut down our H2O instance

In [ ]:
h2o.cluster().shutdown()